# Exploration DVF géolocalisées — 92, année 2024

Source : `https://files.data.gouv.fr/geo-dvf/latest/csv/2024/departements/92.csv.gz` (téléchargé par `scripts/01_download_dvf.py`).

Objectif : chiffrer les pièges avant d'écrire les règles de nettoyage. Aucun filtre n'est appliqué ici.

In [1]:
import duckdb
con = duckdb.connect()
con.sql("""create table d as select * from read_csv('../data/raw/dvf/2024/92.csv.gz',
  types={'id_parcelle':'VARCHAR','code_commune':'VARCHAR','code_postal':'VARCHAR','numero_disposition':'VARCHAR'})""")
q = lambda s: con.sql(s).df()

## 1. Schéma et volumétrie

In [2]:
q('describe d')[['column_name', 'column_type']]

,column_name,column_type
0,id_mutation,VARCHAR
1,date_mutation,DATE
2,numero_disposition,VARCHAR
3,nature_mutation,VARCHAR
4,valeur_fonciere,DOUBLE
5,adresse_numero,BIGINT
6,adresse_suffixe,VARCHAR
7,adresse_nom_voie,VARCHAR
8,adresse_code_voie,VARCHAR
9,code_postal,VARCHAR


In [3]:
q('select count(*) lignes, count(distinct id_mutation) mutations, min(date_mutation) dmin, max(date_mutation) dmax, count(distinct code_commune) communes from d')

,lignes,mutations,dmin,dmax,communes
0,58663,22198,2024-01-02,2024-12-31,36


## 2. Valeurs manquantes (% de lignes)

In [4]:
q('''select column_name, round(100.0 * n_null / (select count(*) from d), 1) pct_null
from (unpivot (select count(*) - count(columns(*)) from d) on columns(*) into name column_name value n_null)
order by 2 desc''')

,column_name,pct_null
0,lot5_surface_carrez,100.0
1,ancien_code_commune,100.0
2,ancien_id_parcelle,100.0
3,ancien_nom_commune,100.0
4,lot4_surface_carrez,99.9
5,lot5_numero,99.7
6,nature_culture_speciale,99.5
7,code_nature_culture_speciale,99.5
8,lot3_surface_carrez,99.4
9,numero_volume,99.4


## 3. Natures de mutation et types de local

In [5]:
q('select nature_mutation, count(*) lignes, count(distinct id_mutation) mutations from d group by 1 order by 2 desc')

,nature_mutation,lignes,mutations
0,Vente,50349,19362
1,Vente en l'état futur d'achèvement,7924,2678
2,Echange,205,91
3,Adjudication,169,61
4,Vente terrain à bâtir,16,7


In [6]:
q("select coalesce(type_local, '<vide>') type_local, count(*) n from d group by 1 order by 2 desc")

,type_local,n
0,Dépendance,29219
1,Appartement,18994
2,<vide>,5922
3,Local industriel. commercial ou assimilé,2421
4,Maison,2107


## 4. Structure d'une mutation

`valeur_fonciere` est le prix **total** de la mutation, répété sur chaque ligne (vérifié : une seule valeur par mutation).

In [7]:
q('select n lignes_par_mutation, count(*) mutations from (select id_mutation, count(*) n from d group by 1) group by 1 order by 1 limit 12')

,lignes_par_mutation,mutations
0,1,6932
1,2,8560
2,3,4788
3,4,1146
4,5,300
5,6,114
6,7,72
7,8,42
8,9,20
9,10,18


In [8]:
q('select count(*) filter (where nv > 1) mutations_valeurs_multiples from (select id_mutation, count(distinct valeur_fonciere) nv from d group by 1)')

,mutations_valeurs_multiples
0,0


In [9]:
con.sql("""create table m as select id_mutation, any_value(nature_mutation) nature, any_value(valeur_fonciere) valeur,
  count(*) filter (where type_local = 'Appartement') n_app,
  count(*) filter (where type_local = 'Dépendance') n_dep,
  count(*) filter (where type_local in ('Maison', 'Local industriel. commercial ou assimilé')) n_autre,
  sum(surface_reelle_bati) filter (where type_local = 'Appartement') surf
from d where nature_mutation in ('Vente', $$Vente en l'état futur d'achèvement$$) group by 1""")
q('''select nature, case when n_app = 0 then '0 appart' when n_app = 1 then '1 appart'
  when n_app <= 5 then '2-5 apparts' else '6+ apparts' end cat, count(*) mutations
from m group by all order by all''')

,nature,cat,mutations
0,Vente,0 appart,5632
1,Vente,1 appart,13184
2,Vente,2-5 apparts,455
3,Vente,6+ apparts,91
4,Vente en l'état futur d'achèvement,0 appart,1482
5,Vente en l'état futur d'achèvement,1 appart,1183
6,Vente en l'état futur d'achèvement,2-5 apparts,6
7,Vente en l'état futur d'achèvement,6+ apparts,6


## 5. Effet des dépendances sur le prix/m²

Mutations à un seul appartement, sans maison ni local commercial.

In [10]:
con.sql('create table s as select *, valeur / surf pm2 from m where n_app = 1 and n_autre = 0 and surf > 0')
q('''select nature, n_dep, count(*) n, round(median(pm2)) med_pm2, round(median(surf)) med_surf, round(median(valeur)) med_valeur
from s group by all having count(*) >= 20 order by all''')

,nature,n_dep,n,med_pm2,med_surf,med_valeur
0,Vente,0,2155,6535.0,32.0,200100.0
1,Vente,1,6109,6311.0,51.0,295000.0
2,Vente,2,3867,6245.0,67.0,391000.0
3,Vente,3,794,6517.0,87.0,545875.0
4,Vente,4,143,7281.0,103.0,731194.0
5,Vente en l'état futur d'achèvement,0,181,7100.0,46.0,318742.0
6,Vente en l'état futur d'achèvement,1,638,6904.0,65.0,419456.0
7,Vente en l'état futur d'achèvement,2,310,7440.0,81.0,555000.0
8,Vente en l'état futur d'achèvement,3,45,8093.0,87.0,715000.0


## 6. Valeurs aberrantes

In [11]:
q('select nature, list_transform(quantile_cont(pm2, [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]), x -> round(x)) quantiles_pm2 from s group by 1')

,nature,quantiles_pm2
0,Vente en l'état futur d'achèvement,"[2084.0, 3661.0, 4475.0, 7143.0, 10533.0, 1274..."
1,Vente,"[3.0, 561.0, 3285.0, 6347.0, 10565.0, 13697.0,..."


In [12]:
q('''select count(*) filter (where valeur < 1000) valeur_lt_1000, count(*) filter (where pm2 < 1500) pm2_lt_1500,
  count(*) filter (where pm2 > 20000) pm2_gt_20000, count(*) filter (where surf < 9) surf_lt_9,
  count(*) filter (where valeur is null) valeur_null from s''')

,valeur_lt_1000,pm2_lt_1500,pm2_gt_20000,surf_lt_9,valeur_null
0,20,182,11,10,11


## 7. Lignes Appartement dupliquées

Un même local peut apparaître plusieurs fois, une fois par nature de culture de la parcelle.

In [13]:
q('''select id_mutation, id_parcelle, lot1_numero, surface_reelle_bati, nature_culture, count(*) n
from d where type_local = 'Appartement' group by all having count(*) > 1 limit 5''')

,id_mutation,id_parcelle,lot1_numero,surface_reelle_bati,nature_culture,n
0,2024-1109135,92050000AU0003,<NA>,27,sols,2
1,2024-1109173,920510000R0053,<NA>,32,sols,2
2,2024-1109173,920510000R0053,<NA>,73,terrains d'agrément,2
3,2024-1109173,920510000R0053,<NA>,98,terrains d'agrément,2
4,2024-1109173,920510000R0053,<NA>,165,sols,2


In [14]:
q("select count(*) apparts_sans_coordonnees from d where type_local = 'Appartement' and latitude is null")

,apparts_sans_coordonnees
0,142
